# VoxEmotion: Speech Emotion Recognition using Librosa & Neural Networks

This notebook guides through an end-to-end machine learning pipeline for recognizing human emotions from raw speech audio signals using acoustic digital signal processing (DSP) and a Multi-Layer Perceptron (MLP) neural network.

### Pipeline Workflow:
1. **Audio Loading & Signal Exploration**: Inspect waveforms, autocorrelations, and spectral energy.
2. **Acoustic Feature Extraction**: Extract 180 low-level acoustic descriptors (40 MFCCs + 12 Chroma + 128 Mel-Bands).
3. **Data Preparation**: Label mapping and train/test stratified splitting.
4. **Model Architecture**: Train a Scaled Multi-Layer Perceptron (MLP) with adaptive learning rates.
5. **Evaluation & Diagnostics**: Accuracy metrics, confusion matrix, and training loss curve.

## Task 1: Import Modules

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('All libraries imported successfully.')

## Task 2: Load Audio and Plot Autocorrelation

In [ ]:
# Load local sample audio file
sample_path = 'samples/sample_happy.wav'
y, sr = librosa.load(sample_path, sr=22050)
print(f'Loaded {sample_path}: {len(y)} samples at {sr} Hz ({len(y)/sr:.2f} seconds)')

# Compute onset strength and autocorrelation
onset_strength = librosa.onset.onset_strength(y=y, sr=sr, hop_length=512)
correlation = librosa.autocorrelate(onset_strength, max_size=4 * sr // 512)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(correlation, color='#6366F1', lw=2)
ax.set(title='Audio Signal Onset Autocorrelation', xlabel='Lag (frames)', ylabel='Correlation')
plt.tight_layout()
plt.show()

## Task 3: Visualize Audio Waveforms Across Emotions

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 6), sharex=True)

# Happy Waveform
librosa.display.waveshow(y, sr=sr, ax=axes[0], color='#10B981')
axes[0].set(title='Happy Emotion Waveform (Active Cadence & High Pitch Variation)')

# Calm Waveform
y_calm, _ = librosa.load('samples/sample_calm.wav', sr=22050)
librosa.display.waveshow(y_calm, sr=sr, ax=axes[1], color='#06B6D4')
axes[1].set(title='Calm Emotion Waveform (Smooth Envelope & Low Frequency)')

plt.tight_layout()
plt.show()

## Task 4: Acoustic Feature Extraction (180 Dimensions)

In [ ]:
def extract_feature(file_name):
    """Extract 180 low-level acoustic descriptors: MFCC (40) + Chroma (12) + MelSpectrogram (128)."""
    with sf.SoundFile(file_name) as sound_file:
        audio = sound_file.read(dtype='float32')
        sample_rate = sound_file.samplerate
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)

        result = np.array([])
        # 1. 40 Mel-Frequency Cepstral Coefficients
        mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40).T, axis=0)
        result = np.hstack((result, mfcc))

        # 2. 12 Chroma STFT Semitone Pitch Bins
        stft = np.abs(librosa.stft(audio))
        chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
        result = np.hstack((result, chroma))

        # 3. 128 Mel-scale Spectrogram Frequency Bands
        mel = np.mean(librosa.feature.melspectrogram(y=audio, sr=sample_rate, n_mels=128).T, axis=0)
        result = np.hstack((result, mel))

    return result

feat_sample = extract_feature('samples/sample_happy.wav')
print(f'Extracted feature vector dimension: {len(feat_sample)} (40 MFCC + 12 Chroma + 128 Mel)')

## Task 5: Define Emotion Classes

In [ ]:
emotions = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '07': 'disgust',
}
observed_emotions = ['neutral', 'calm', 'happy', 'sad', 'angry', 'disgust']
print('Observed target emotions:', observed_emotions)

## Task 6: Load and Prepare Training Corpus

In [ ]:
from src.voxemotion.dataset import build_benchmark_dataset

# Load balanced acoustic corpus across emotions
X, y = build_benchmark_dataset(num_samples_per_emotion=40, observed_emotions=observed_emotions)
print(f'Corpus Shape: {X.shape[0]} samples with {X.shape[1]} features each across {len(set(y))} emotions.')

## Task 7: Split Training and Testing Partitions

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training Set: {len(X_train)} samples')
print(f'Testing Set:  {len(X_test)} samples')

## Task 8: Visualize Emotion Class Distribution

In [ ]:
df_dist = pd.DataFrame(y_test, columns=['Emotion'])
plt.figure(figsize=(8, 4))
df_dist['Emotion'].value_counts().plot(kind='bar', color='#8B5CF6')
plt.title('Test Set Emotion Distribution')
plt.xlabel('Emotion Category')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Task 9: Initialize and Train Multi-Layer Perceptron (MLP) Classifier

In [ ]:
# Standardize acoustic features (zero mean, unit variance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Neural Classifier with L2 Regularization & Adaptive Learning Rate
model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    alpha=0.01,
    batch_size=32,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=300,
    random_state=42,
)

model.fit(X_train_scaled, y_train)
print(f'Model converged in {model.n_iter_} iterations.')

## Task 10: Model Evaluation & Confusion Matrix

In [ ]:
predictions = model.predict(X_test_scaled)
acc = accuracy_score(y_test, predictions)
print(f'Test Set Accuracy: {acc*100:.2f}%\n')
print(classification_report(y_test, predictions))

# Confusion Matrix
cm = confusion_matrix(y_test, predictions, labels=model.classes_)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(model.classes_)))
ax.set_yticks(range(len(model.classes_)))
ax.set_xticklabels(model.classes_, rotation=45)
ax.set_yticklabels(model.classes_)
ax.set_title(f'Confusion Matrix (Accuracy: {acc*100:.1f}%)')
for i in range(len(model.classes_)):
    for j in range(len(model.classes_)):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.tight_layout()
plt.show()

## Task 11: Plot Training Loss Curve

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(model.loss_curve_, color='#EC4899', lw=2)
plt.title('MLP Neural Classifier Loss Curve')
plt.xlabel('Epoch Iteration')
plt.ylabel('Cross-Entropy Loss')
plt.tight_layout()
plt.show()

## Task 12: Predict on Audio Files

In [ ]:
for test_file in ['samples/sample_happy.wav', 'samples/sample_angry.wav', 'samples/sample_calm.wav']:
    feat = extract_feature(test_file)
    feat_scaled = scaler.transform(feat.reshape(1, -1))
    pred = model.predict(feat_scaled)[0]
    prob = np.max(model.predict_proba(feat_scaled)[0])
    print(f'{Path(test_file).name:<18} -> Predicted: {pred.upper():<8} ({prob*100:.1f}% Confidence)')